# Baseline Scientific Audit (s2audit)

Runs the rigorous pre-TCR-Net audit of the trained reference-U-Net baseline:
ground-truth cloud-quality, corrected micro/macro metrics, per-band + vegetation-index
accuracy, non-learned baseline comparison, stratified evaluation, and a data-leakage check.

**Inputs to attach (right panel -> Add Input):**
1. Your **synthetic dataset** (contains `samples/`, `patch_library/`, `_gt_patches.jsonl`).
2. Your training **notebook output** (contains `best.pt`).

Nothing is retrained; the checkpoint and dataset are never modified. Outputs go to `/kaggle/working/audit`.

In [ ]:
# 1) Get the code (GitHub). Re-run safe.
REPO_URL = "https://github.com/Yuv1ne04/trial3kaggle.git"
import os
!rm -rf /kaggle/working/trial3
!git clone -q $REPO_URL /kaggle/working/trial3
%cd /kaggle/working/trial3
!git log --oneline -1
!pip -q install rasterio >/dev/null 2>&1
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2) Locate dataset root and best.pt under /kaggle/input (dir-only walk; prune huge folders)
import os
PRUNE = {"patch_library","cloud_tile_library","mask_library","reference_library",
         "target_library","samples","train","validation","test",".git"}

def find_dataset_root(base="/kaggle/input"):
    for dirpath, dirnames, filenames in os.walk(base):
        if "samples" in dirnames and (os.path.isdir(os.path.join(dirpath,"samples","test"))
                                      or os.path.isdir(os.path.join(dirpath,"samples","train"))):
            return dirpath
        dirnames[:] = [d for d in dirnames if d not in PRUNE]
    return None

def find_checkpoint(bases=("/kaggle/input","/kaggle/working")):
    best=None
    for base in bases:
        for dirpath, dirnames, filenames in os.walk(base):
            for f in filenames:
                if f in ("best.pt","latest.pt"):
                    p=os.path.join(dirpath,f)
                    if f=="best.pt": return p
                    best=best or p
            dirnames[:] = [d for d in dirnames if d not in PRUNE]
    return best

DATA_ROOT = find_dataset_root()
CKPT = find_checkpoint()
print("DATA_ROOT:", DATA_ROOT)
print("CKPT     :", CKPT)
assert DATA_ROOT and CKPT, "Could not locate dataset root or checkpoint - check attached inputs.

## Quick smoke (a few minutes) — verify everything runs before the full pass
Uses small caps on GPU. Confirms the pipeline, then Cell 4 does the real full run.

In [ ]:
!python -m s2audit --dataset "$DATA_ROOT" --checkpoint "$CKPT"     --output /kaggle/working/audit_smoke     --max-samples 256 --gt-max-patches 1000 --leakage-max-samples 8000     --device auto --batch-size 16 --num-workers 2 --visualize-n 8

## Full audit — whole test split, all unique GT patches, full leakage scan
`--full` ignores all caps. On a T4 this is the authoritative run. (The 89k-manifest
leakage scan and full GT audit dominate the time; batched GPU inference handles the eval.)

In [ ]:
!python -m s2audit --dataset "$DATA_ROOT" --checkpoint "$CKPT"     --output /kaggle/working/audit     --full --device auto --batch-size 32 --num-workers 2 --visualize-n 12

In [ ]:
# 5) Show the verdict
import json
s = json.load(open("/kaggle/working/audit/scientific_audit_summary.json"))
print("OVERALL:", s["overall_status"], "
")
for k,v in s["scientific_answers"].items():
    print(f"[{k}]
   {v}
")
print("RECOMMENDATION:", s["recommendation"])

In [ ]:
# 6) Key tables
import pandas as pd, glob
for name in ["baseline_comparison","test_metrics","vegetation_metrics","stratified_metrics","per_band_metrics"]:
    p=f"/kaggle/working/audit/{name}.csv"
    try:
        print("
==== ", name, " ====")
        display(pd.read_csv(p))
    except Exception as e:
        print(name, "missing:", e)

In [ ]:
# 7) Look at an enhanced reconstruction panel and a GT-audit panel
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/kaggle/working/audit/visual_reconstruction/*.png"))[:2]:
    display(Image(p))
for p in sorted(glob.glob("/kaggle/working/audit/visual_ground_truth_audit/*.png")):
    display(Image(p))

## Outputs written to `/kaggle/working/audit/`
- `scientific_audit_summary.json` — overall verdict + answers
- `ground_truth_quality.csv`, `ground_truth_filter_manifest.json`, `visual_ground_truth_audit/`
- `test_evaluation_report.json`, `test_metrics.csv`, `per_band_metrics.csv`,
  `vegetation_metrics.csv`, `stratified_metrics.csv`, `baseline_comparison.csv`
- `data_leakage_audit.json`
- `visual_reconstruction/` — enhanced dual-stretch + NDVI panels

To retrain on clean ground truth later, point the dataset at the filter manifest:
`SyntheticDataset(root, split='train', gt_filter='/kaggle/working/audit/ground_truth_filter_manifest.json')`.